In [13]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np


In [21]:
column_headers = ["date","timestamp","open","high","low","close","volume"]
filename = "/mnt/c/Shaukat/code_repo/HighFrequencyTradingCoding/data/DAT_MT_AUDUSD_M1_202411.csv"
df = pd.read_csv(filename, names=column_headers)

# combine dates and timestamps into single datetime column
df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['timestamp'], format='%Y.%m.%d %H:%M')

# drop date, timestamp and volume
df.drop(columns=["date", "timestamp", "volume"], inplace=True)

# Setting datetime as index
df.set_index("datetime", inplace=True)

# Sort dataframe by datetime
df.sort_values(by="datetime", inplace=True)

In [22]:
df.head(5)

,open,high,low,close
datetime,,,,
2024-11-01 00:00:00,0.65759,0.65767,0.65757,0.65758
2024-11-01 00:01:00,0.65759,0.65759,0.65750,0.65754
2024-11-01 00:02:00,0.65754,0.65764,0.65754,0.65761
2024-11-01 00:03:00,0.65760,0.65760,0.65751,0.65751
2024-11-01 00:04:00,0.65753,0.65758,0.65751,0.65755


In [23]:
# Add last lag_minutes minutes
flag_augment_data = False
if flag_augment_data:
    lag_minutes = 3
    for iter_lag in range(1,lag_minutes):
        # print(iter_lag)
        df[f"open_t-{iter_lag}"] = df["open"].shift(iter_lag)
        df[f"open_avg_t-{iter_lag}"] = df["open"].rolling(window=iter_lag).mean()
        df[f"high_t-{iter_lag}"] = df["high"].shift(iter_lag)
        df[f"low_t-{iter_lag}"] = df["low"].shift(iter_lag)
        df[f"close_t-{iter_lag}"] = df["close"].shift(iter_lag)
    # df["open_t-1"] = df["open"].shift(1)

In [24]:
df.head(5)

,open,high,low,close
datetime,,,,
2024-11-01 00:00:00,0.65759,0.65767,0.65757,0.65758
2024-11-01 00:01:00,0.65759,0.65759,0.65750,0.65754
2024-11-01 00:02:00,0.65754,0.65764,0.65754,0.65761
2024-11-01 00:03:00,0.65760,0.65760,0.65751,0.65751
2024-11-01 00:04:00,0.65753,0.65758,0.65751,0.65755


In [25]:
# Add price label
df['label'] = df.apply(lambda row: 1 if row['close'] > row['open'] else 0, axis=1)


In [26]:
df.head(5)

,open,high,low,close,label
datetime,,,,,
2024-11-01 00:00:00,0.65759,0.65767,0.65757,0.65758,0
2024-11-01 00:01:00,0.65759,0.65759,0.65750,0.65754,0
2024-11-01 00:02:00,0.65754,0.65764,0.65754,0.65761,1
2024-11-01 00:03:00,0.65760,0.65760,0.65751,0.65751,0
2024-11-01 00:04:00,0.65753,0.65758,0.65751,0.65755,1


In [27]:
# Remove Nans
print(f"len_df: {len(df)} before removing NA")
df.dropna(inplace=True)
print(f"len_df: {len(df)} after removing NA")

len_df: 29658 before removing NA
len_df: 29658 after removing NA


In [28]:
df.head(12)

,open,high,low,close,label
datetime,,,,,
2024-11-01 00:00:00,0.65759,0.65767,0.65757,0.65758,0
2024-11-01 00:01:00,0.65759,0.65759,0.65750,0.65754,0
2024-11-01 00:02:00,0.65754,0.65764,0.65754,0.65761,1
2024-11-01 00:03:00,0.65760,0.65760,0.65751,0.65751,0
2024-11-01 00:04:00,0.65753,0.65758,0.65751,0.65755,1
2024-11-01 00:05:00,0.65755,0.65761,0.65752,0.65754,0
2024-11-01 00:06:00,0.65753,0.65760,0.65753,0.65755,1
2024-11-01 00:07:00,0.65754,0.65757,0.65753,0.65756,1
2024-11-01 00:08:00,0.65757,0.65757,0.65748,0.65750,0


In [29]:
print(df.index.min(), df.index.max())

2024-11-01 00:00:00 2024-11-29 16:58:00


# LSTM Model


In [30]:
# Testing the inputs of LSTM Layer
# Sequence looks like: x_1 -> x_2 where x_1,x_2 in R^4
input_dim = 4
len_of_seq = 2
dim_hidden_state = 5
batch_size = 2 

'''
- initialize random inputs for testing
- Input in LSTM is passed as (batch_size, len_of_seq, input_dim)
- From Pytorch documentation, LSTM expects the input as (N,L,H_in) when batch_first=True
where N=batch size, L=sequence length and H_in=input_size
'''
input_to_lstm = torch.rand(batch_size, len_of_seq, input_dim)
print(input_to_lstm)
print(input_to_lstm.shape)
print('\n\n')
print(input_to_lstm[0, :, :].shape)

tensor([[[0.6811, 0.2288, 0.0063, 0.8428],
         [0.2225, 0.3599, 0.2580, 0.7626]],

        [[0.9885, 0.8548, 0.2402, 0.1804],
         [0.1647, 0.4640, 0.1789, 0.9459]]])
torch.Size([2, 2, 4])



torch.Size([2, 4])


In [31]:
# Initialize LSTM layer
lstm_layer = nn.LSTM(input_size=input_dim, hidden_size=dim_hidden_state, num_layers=1, batch_first=True)

# pass the input and extract output (Forward pass)
output, (h_n, c_n) = lstm_layer(input_to_lstm)

In [32]:
# Inspect the output from LSTM layer
print("LSTM Output Shape:", output.shape)  # (batch_size=2, seq_len=2, hidden_size=5)
print("h_n Shape:", h_n.shape)            # (num_layers=1, batch_size=2, hidden_size=5)
print("c_n Shape:", c_n.shape)            # (num_layers=1, batch_size=2, hidden_size=5)
print('\n')
print('printing output')
print(output)
print('\n')
print('printing h_n')
print(h_n)

LSTM Output Shape: torch.Size([2, 2, 5])
h_n Shape: torch.Size([1, 2, 5])
c_n Shape: torch.Size([1, 2, 5])


printing output
tensor([[[-0.1719, -0.0827, -0.0870, -0.0741,  0.1490],
         [-0.2579, -0.1172, -0.1213, -0.0449,  0.1983]],

        [[-0.2040, -0.1632, -0.0744, -0.2097,  0.0449],
         [-0.2726, -0.1757, -0.1322, -0.0865,  0.2151]]],
       grad_fn=<TransposeBackward0>)


printing h_n
tensor([[[-0.2579, -0.1172, -0.1213, -0.0449,  0.1983],
         [-0.2726, -0.1757, -0.1322, -0.0865,  0.2151]]],
       grad_fn=<StackBackward0>)


In [ ]:
# Prepare DATASET and DATALOADER
# https://pytorch.org/tutorials/beginner/basics/data_tutorial.html
# A custom Dataset class must implement three functions: __init__, __len__, and __getitem__.

# Now proceed to prepare dataloader and dataset class 
        